<a href="https://colab.research.google.com/github/sakshumvij/SPECTRA-Skin-Pathology-Evaluation-via-Convolutional-Training-and-Recognition-Algorithms/blob/main/SPECTRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Imports
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.metrics import AUC

#Configuration
IMG_SIZE = (299, 299)
BATCH_SIZE = 32
EPOCHS = 25
N_FOLDS = 5
SEED = 42

# CHANGE THIS AS NEEDED
DATA_DIR = os.environ.get("DATA_DIR", "./dataset")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

#Dataframe construction
def build_df(root):
    filepaths, labels = [], []

    for label in ["benign", "malignant"]:
        class_dir = os.path.join(root, label)
        for f in os.listdir(class_dir):
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                filepaths.append(os.path.join(class_dir, f))
                labels.append(label)

    df = pd.DataFrame({"filepath": filepaths, "label": labels})
    df["label_int"] = (df["label"] == "malignant").astype(int)
    return df

df = build_df(TRAIN_DIR)

# Data generation
train_datagen = ImageDataGenerator(
    # Augmentation
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.15,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

test_gen = val_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

# Model builder
def build_model():
    # Base model (transfer learning)
    base = InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )

    # Freeze most layers (other than top 250 trainable)
    for layer in base.layers[:-250]:
        layer.trainable = False

    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dense(512, activation="relu"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=["accuracy", AUC(name="auc")]
    )

    return model

# 5-Fold Cross Validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

test_preds = []
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label_int"]), 1):
    print(f"\nFold {fold}/{N_FOLDS}")

    tf.keras.backend.clear_session()

    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()

    train_gen = train_datagen.flow_from_dataframe(
        train_df,
        x_col="filepath",
        y_col="label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        shuffle=True,
        seed=SEED
    )

    val_gen = val_datagen.flow_from_dataframe(
        val_df,
        x_col="filepath",
        y_col="label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        shuffle=False
    )

    # Class weights
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(train_df["label_int"]),
        y=train_df["label_int"]
    )
    class_weights = dict(enumerate(class_weights))

    model = build_model()

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=[
            EarlyStopping(
                monitor="val_auc",
                mode="max",
                patience=5,
                restore_best_weights=True
            )
        ],
        verbose=1
    )

    # Save one fold
    save_path = f"./models/Fold{fold}.keras" #alter to proper file path
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    model.save(save_path)

    # Predict on test set
    test_gen.reset()
    preds = model.predict(test_gen, verbose=0).ravel()
    test_preds.append(preds)

    best_auc = max(history.history["val_auc"])
    fold_aucs.append(best_auc)

#Ensembled Evaluation
ensemble_preds = np.mean(test_preds, axis=0)
y_true = test_gen.classes

auc_metric = AUC()
auc_metric.update_state(y_true, ensemble_preds)

print(f"\nEnsembled Test AUC: {auc_metric.result().numpy():.4f}")
print(f"Fold AUCs: {fold_aucs}")

In [ ]:
#Saliency Mapping
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2


IMG_PATH = "example.jpg"
IMG_SIZE = (299, 299)
LAST_CONV = "mixed10" # last convolutional layer of InceptionV3

#Prepare image
img_orig = cv2.imread(IMG_PATH)
img_orig = cv2.resize(img_orig, IMG_SIZE)
img_rgb = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)

img_array = img_rgb.astype("float32") / 255.0
img_array = np.expand_dims(img_array, axis=0)

base_model = model.layers[0]
inner_model = tf.keras.models.Model(
    inputs=base_model.inputs,
    outputs=[base_model.get_layer(LAST_CONV).output, base_model.output]
)

def get_predictions(input_tensor):
    conv_out, x = inner_model(input_tensor)
    for layer in model.layers[1:]:
        x = layer(x)
    return conv_out, x

#Compute Saliency Map
img_tensor = tf.convert_to_tensor(img_array)
with tf.GradientTape() as tape:
    tape.watch(img_tensor)
    _, preds = get_predictions(img_tensor)
    score = preds[:, tf.argmax(preds[0])]

# Get raw gradients
grads_pixels = tape.gradient(score, img_tensor)
saliency_raw = tf.reduce_max(tf.abs(grads_pixels), axis=-1)[0].numpy()

# Reduce Noise

# Gaussian Blur
saliency_blurred = cv2.GaussianBlur(saliency_raw, (11, 11), 0)

# Percentile threshhold (85%)
threshold_value = np.percentile(saliency_blurred, 85)
saliency_blurred[saliency_blurred < threshold_value] = 0

# Normalize from [0,1]
saliency_norm = (saliency_blurred - np.min(saliency_blurred)) / (np.max(saliency_blurred) - np.min(saliency_blurred) + 1e-10)

saliency_img = np.uint8(255 * saliency_norm)

#Show results
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(img_rgb)
ax[0].set_title("Original Image")

ax[1].imshow(saliency_img, cmap='hot')
ax[1].set_title("Saliency Map (Denoised)")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()